In [26]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

In [2]:
SEED: int = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data pipeline

In [8]:
def load_dataset(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def filter_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes network-specific socket features and junk columns
    to prevent model overfitting.
    """
    df.columns = df.columns.str.strip()
    columns_to_drop: list[str] = [
        'Unnamed: 0',
        'Flow ID',
        'Source IP',
        'Source Port',
        'Destination IP',
        'Destination Port',
        'Timestamp',
        'Fwd Header Length.1',
        'SimillarHTTP',
        'Inbound'
    ]
    existing_columns_to_drop: list[str] = [col for col in columns_to_drop if col in df.columns]
    filtered_df: pd.DataFrame = df.drop(existing_columns_to_drop, axis=1)

    return filtered_df

def clean_missing_and_infinite_values(df: pd.DataFrame) -> pd.DataFrame:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(axis=0, how="any", inplace=True)

    return df

In [9]:
from sklearn.preprocessing import LabelEncoder

def encode_multiclass_labels(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    Encodes text labels to integers (0, 1, 2...).
    Returns the updated DataFrame and the class mapping dictionary.
    """
    df["Label"] = df["Label"].astype(str).str.strip().str.upper()
    label_encoder = LabelEncoder()
    df["Label"] = label_encoder.fit_transform(df["Label"])
    encoded_keys = label_encoder.transform(label_encoder.classes_)
    class_names = label_encoder.classes_
    mapping = {int(k): str(v) for k, v in zip(encoded_keys, class_names)}

    return df, mapping


def build_balanced_multiclass_dataset(directory_path: str, global_max_per_class: int = 20_000) -> tuple[pd.DataFrame, dict]:
    """
    Creates multi-class dataset, taking as a whole underrepresented classes.
    Returns the master DataFrame and the class mapping dictionary.
    """
    processed_dfs = []

    for filename in sorted(os.listdir(directory_path)):
        if filename.endswith(".csv"):
            file_path = os.path.join(directory_path, filename)
            print(f"Loading: {filename}...")

            temp_df = load_dataset(file_path)
            temp_df = filter_features(temp_df)
            temp_df = clean_missing_and_infinite_values(temp_df)
            temp_df["Label"] = temp_df["Label"].astype(str).str.strip().str.upper()

            for _, group_df in temp_df.groupby("Label"):
                if len(group_df) > global_max_per_class:
                    processed_dfs.append(group_df.sample(n=global_max_per_class, random_state=SEED))
                else:
                    processed_dfs.append(group_df)

    print("\nMerging into global dataset...")
    master_df = pd.concat(processed_dfs, ignore_index=True)

    master_df = master_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    print("Applying label encoding...")
    master_df, class_mapping = encode_multiclass_labels(master_df)

    return master_df, class_mapping

In [11]:
import json

DATA_DIR: str = "data/01-12"
MASTER_DATASET_PATH: str = "data/master_clean_dataset.csv"
MAPPING_PATH: str = "data/class_mapping.json"

if os.path.exists(MASTER_DATASET_PATH) and os.path.exists(MAPPING_PATH):
    print("Loading cached dataset and mapping...")
    master_df: pd.DataFrame = pd.read_csv(MASTER_DATASET_PATH)
    with open(MAPPING_PATH, "r") as f:
        loaded_mapping = json.load(f)
        class_mapping: dict = {int(k): v for k, v in loaded_mapping.items()}

else:
    print("Building dataset...")
    master_df, class_mapping = build_balanced_multiclass_dataset(DATA_DIR, global_max_per_class=20_000)

    master_df.to_csv(MASTER_DATASET_PATH, index=False)
    print(f"Dataset saved to {MASTER_DATASET_PATH}")
    with open(MAPPING_PATH, "w") as f:
        json.dump(class_mapping, f, indent=4)
    print(f"Class mapping saved to {MAPPING_PATH}")

print("\n--- FINAL MULTI-CLASS DISTRIBUTION ---")
print(master_df["Label"].value_counts().sort_index())

print("\n--- CLASS MAPPING ---")
for encoded_val, original_name in class_mapping.items():
    print(f"Class {encoded_val} -> {original_name}")

print("\nDataset shape:", master_df.shape)


Loading cached dataset and mapping...

--- FINAL MULTI-CLASS DISTRIBUTION ---
Label
0     51330
1     20000
2     20000
3     20000
4     20000
5     20000
6     20000
7     20000
8     20000
9     20000
10    20000
11    20000
12      439
Name: count, dtype: int64

--- CLASS MAPPING ---
Class 0 -> BENIGN
Class 1 -> DRDOS_DNS
Class 2 -> DRDOS_LDAP
Class 3 -> DRDOS_MSSQL
Class 4 -> DRDOS_NETBIOS
Class 5 -> DRDOS_NTP
Class 6 -> DRDOS_SNMP
Class 7 -> DRDOS_SSDP
Class 8 -> DRDOS_UDP
Class 9 -> SYN
Class 10 -> TFTP
Class 11 -> UDP-LAG
Class 12 -> WEBDDOS

Dataset shape: (271769, 78)


In [ ]:
def split_dataframe(df: pd.DataFrame, train_frac: float = 0.7) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X: np.ndarray = df.drop("Label", axis=1).values
    y: np.ndarray = df["Label"].values

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(1.0-train_frac), random_state=SEED, stratify=y
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

def normalize(X_train: np.ndarray, X_val: np.ndarray, X_test: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    scaler: MinMaxScaler = MinMaxScaler(feature_range=(0,1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled

X_train, y_train, X_val, y_val, X_test, y_test = split_dataframe(master_df)
X_train, X_val, X_test = normalize(X_train, X_val, X_test)

In [14]:
class DDOSDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        # self.y = torch.from_numpy(y).long()
        y = torch.from_numpy(y)
        self.y = (y != 0).long()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, index: int):
        return self.X[index], self.y[index]

In [15]:
train_dataset = DDOSDataset(X_train, y_train)
valid_dataset = DDOSDataset(X_val, y_val)
test_dataset = DDOSDataset(X_test, y_test)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [16]:
sample_inputs, sample_labels = next(iter(train_loader))
input_dim = sample_inputs.shape[-1]
input_dim

77

In [17]:
import gc

del master_df
gc.collect()

1593

# Modele

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

print(f"Dostępność GPU: {torch.cuda.is_available()}")

Dostępność GPU: True


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu" )
device

device(type='cuda')

## Klasy modeli

In [20]:
class DDoSNetEncoder(nn.Module):
    def __init__(self, input_dim):
        super(DDoSNetEncoder, self).__init__()
        self.rnn1 = nn.RNN(input_size=input_dim, hidden_size=64, batch_first=True, nonlinearity='relu')
        self.rnn2 = nn.RNN(input_size=64, hidden_size=32, batch_first=True, nonlinearity='relu')
        self.rnn3 = nn.RNN(input_size=32, hidden_size=16, batch_first=True, nonlinearity='relu')
        self.rnn4 = nn.RNN(input_size=16, hidden_size=8, batch_first=True, nonlinearity='relu')

    def forward(self, x):
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        x, _ = self.rnn3(x)
        x, _ = self.rnn4(x)
        return x

In [21]:
class DDoSNetDecoder(nn.Module):
    def __init__(self, output_dim):
        super(DDoSNetDecoder, self).__init__()
        self.rnn1 = nn.RNN(input_size=8, hidden_size=8, batch_first=True, nonlinearity='relu')
        self.rnn2 = nn.RNN(input_size=8, hidden_size=16, batch_first=True, nonlinearity='relu')
        self.rnn3 = nn.RNN(input_size=16, hidden_size=32, batch_first=True, nonlinearity='relu')
        self.rnn4 = nn.RNN(input_size=32, hidden_size=64, batch_first=True, nonlinearity='relu')

        self.reconstruction_layer = nn.Linear(64, output_dim)

    def forward(self, x):
        x, _ = self.rnn1(x)
        x, _ = self.rnn2(x)
        x, _ = self.rnn3(x)
        x, _ = self.rnn4(x)

        reconstructed = self.reconstruction_layer(x)
        return reconstructed

In [22]:
class DDoSNetReconstruction(nn.Module):
    def __init__(self, input_dim, encoder=None, decoder=None):
        super(DDoSNetReconstruction, self).__init__()

        self.encoder = encoder if encoder is not None else DDoSNetEncoder(input_dim)
        self.decoder = decoder if decoder is not None else DDoSNetDecoder(input_dim)

    def forward(self, x):
        encoded_sequence = self.encoder(x)
        reconstructed_sequence = self.decoder(encoded_sequence)

        return reconstructed_sequence

In [ ]:

class DDoSNetClassifier(nn.Module):
    def __init__(self, input_dim, encoder=None, decoder=None):
        super(DDoSNetClassifier, self).__init__()

        self.encoder = encoder if encoder is not None else DDoSNetEncoder(input_dim)
        self.decoder = decoder if decoder is not None else DDoSNetDecoder(input_dim)

        self.classifier = nn.Linear(input_dim, 2)

    def forward(self, x):
        encoded = self.encoder(x)
        reconstructed = self.decoder(encoded)

        last_step_reconstructed = reconstructed[:, -1, :]
        logits = self.classifier(last_step_reconstructed)

        return logits

## Trenowanie

### Fnkcje pomocnicze

In [40]:
def finetune_classifier(model, train_loader, val_loader, device, epochs=50, lr=0.0001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())

    train_losses = [None]*epochs
    val_losses = [None]*epochs
    train_acc = [None]*epochs
    val_acc = [None]*epochs

    model.to(device)

    epoch_bar = tqdm(range(epochs), desc="Trening modelu")
    for epoch in epoch_bar:
        model.train()
        train_loss, train_correct, total = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device).long()
            if inputs.dim() == 2:
                inputs = inputs.unsqueeze(1)

            optimizer.zero_grad()

            logits = model(inputs)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        epoch_train_loss = train_loss / total
        epoch_train_acc = train_correct / total
        train_losses[epoch] = epoch_train_loss
        train_acc[epoch] = epoch_train_acc

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device).long()
                if inputs.dim() == 2:
                    inputs = inputs.unsqueeze(1)

                logits = model(inputs)
                loss = criterion(logits, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(logits.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        val_losses[epoch] = epoch_val_loss
        val_acc[epoch] = epoch_val_acc

        epoch_bar.set_postfix({
            'train_acc': f"{train_acc[epoch]:.4f}", 
            'val_acc': f"{val_acc[epoch]:.4f}"
        })

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_weights)
    return model, (train_losses, val_losses), (train_acc, val_acc)


### Testowanie parametrów

In [31]:
lrs = [0.1, 0.01, 0.001, 0.0001, 0.00005]

In [ ]:
results = []
best_val_acc = 0.0
best_model_weights = None
best_lr = None

In [41]:
loaded_reconstruction_model = DDoSNetReconstruction(input_dim=input_dim)
saved_model_path = "./saved_models/best_autoencoder_full.pth"

loaded_reconstruction_model.load_state_dict(
    torch.load(saved_model_path, map_location=device)
)

pretrained_encoder = loaded_reconstruction_model.encoder
pretrained_decoder = loaded_reconstruction_model.decoder

In [42]:
for i, lr in enumerate(lrs):
    print(f"--- Trening {i+1}/{len(lrs)} | Ustawienia: {lr=} ---")

    current_model = DDoSNetClassifier(
        input_dim=input_dim,
        encoder=copy.deepcopy(pretrained_encoder),
        decoder=copy.deepcopy(pretrained_decoder)
    ).to(device)
    
    trained_model, (train_losses, val_losses), (train_acc, val_acc) = finetune_classifier(
        model=current_model, 
        train_loader=train_loader, 
        val_loader=val_loader, 
        device=device, 
        epochs=10, 
        lr=lr
    )
    
    current_best_acc = max(val_acc) if len(val_acc) > 0 else 0
    current_min_loss = min(val_losses) if len(val_losses) > 0 else 0
    
    results.append({
        'lr': lr,
        'best_val_loss': current_min_loss,
        'best_val_acc': current_best_acc
    })
    
    if current_best_acc > best_val_acc:
        best_val_acc = current_best_acc
        best_lr = lr
        best_model_weights = copy.deepcopy(trained_model.state_dict())

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='best_val_acc', ascending=False).reset_index(drop=True)

print("\n================ PODSUMOWANIE WYNIKÓW ================")
print(results_df.to_string())
print("======================================================")

print(f"\nNajlepszy lr to: {best_lr}")
print(f"Uzyskana dokładność na zbiorze walidacyjnym: {best_val_acc:.4f}")

--- Trening 1/5 | Ustawienia: lr=0.1 ---


Trening modelu:   0%|          | 0/10 [00:00<?, ?it/s]

Trening modelu: 100%|██████████| 10/10 [04:20<00:00, 26.05s/it, train_acc=0.8096, val_acc=0.8111]


--- Trening 2/5 | Ustawienia: lr=0.01 ---


Trening modelu: 100%|██████████| 10/10 [04:24<00:00, 26.44s/it, train_acc=0.9520, val_acc=0.9508]


...

--- Trening 4/5 | Ustawienia: lr=0.0001 ---


Trening modelu: 100%|██████████| 10/10 [07:40<00:00, 46.01s/it, train_acc=0.9973, val_acc=0.9972]


--- Trening 5/5 | Ustawienia: lr=5e-05 ---


Trening modelu: 100%|██████████| 10/10 [06:38<00:00, 39.85s/it, train_acc=0.9969, val_acc=0.9969]


================ PODSUMOWANIE WYNIKÓW ================
        lr  best_val_loss  best_val_acc
0  0.00100       0.009432      0.997571
1  0.00010       0.011473      0.997302
2  0.00005       0.012653      0.996958
3  0.01000       0.015648      0.996860
4  0.01000       0.484570      0.811137
5  0.10000       0.484658      0.811137
6  0.10000       0.484791      0.811137
7  0.00005       0.484570      0.811137
8  0.00010       0.484570      0.811137
9  0.00100       0.484573      0.811137

Najlepszy lr to: 0.001
Uzyskana dokładność na zbiorze walidacyjnym: 0.9976
